# Solutions — Concurrency: Threads, Locks, Processes, asyncio, Producer–Consumer

Exercise notebook: `10_advanced_python/03_concurrency_basics.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))
from labkit import workers      # sha256_path, count_primes, line_count

## Exercise 1 — 🟢 Easy · Ordered parallel map with threads

**Working solution**

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def thread_map(func, items, max_workers=8):
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        return list(pool.map(func, items))

**Explanation**

`executor.map` preserves input order and re-raises a worker's exception when you reach that result. The `with` block waits for all work to finish.

**Complexity:** ≈ total_io_time / workers.

**Common mistakes**

- Creating one thread per item manually.
- Using threads for CPU-heavy pure-Python work and expecting speed-ups.

## Exercise 2 — 🟡 Medium · Race conditions and locks

**Working solution**

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

class SafeCounter:
    def __init__(self):
        self._counts = {}
        self._lock = threading.Lock()

    def add(self, key, n=1):
        with self._lock:
            self._counts[key] = self._counts.get(key, 0) + n

    def snapshot(self):
        with self._lock:
            return dict(self._counts)

def count_events_concurrently(batches, workers=8):
    counter = SafeCounter()
    def work(batch):
        for event_type in batch:
            counter.add(event_type)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(work, batches))
    return counter.snapshot()

**Explanation**

Read-modify-write (`get` then set) is not atomic: two threads can read the same old value. A `Lock` used as a context manager makes the critical section exclusive.
Even better design: let each worker build a local `Counter` and merge results at the end — no shared state at all (that's how Spark aggregates).

**Complexity:** O(total events).

**Common mistakes**

- Believing the GIL makes `d[k] += 1` safe — it doesn't make multi-step operations atomic.

## Exercise 3 — 🟡 Medium · CPU-bound work with processes

**Working solution**

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from labkit.workers import sha256_path

def parallel_checksums(paths, workers=4):
    paths = [str(p) for p in paths]
    if not paths:
        return {}
    with ProcessPoolExecutor(max_workers=workers) as pool:
        return dict(pool.map(sha256_path, paths, chunksize=max(1, len(paths) // (workers * 4))))

**Explanation**

Each process has its own interpreter and GIL, so CPU-bound work truly runs in parallel. Arguments and results are pickled between processes, so the worker must be importable
and the data passed should be small (paths, not file contents). On Windows/macOS, scripts using process pools need an `if __name__ == "__main__":` guard.

**Threads vs processes benchmark**

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from labkit.workers import count_primes

for Pool in (ThreadPoolExecutor, ProcessPoolExecutor):
    start = time.perf_counter()
    with Pool(max_workers=4) as pool:
        list(pool.map(count_primes, [60_000] * 8))
    print(Pool.__name__, round(time.perf_counter() - start, 2), "s")

**Complexity:** ≈ total_cpu / cores (+ process start-up overhead).

**Common mistakes**

- Passing a lambda or a notebook-defined function to a process pool → `PicklingError`.
- Using processes for tiny tasks (overhead dominates).

## Exercise 4 — 🟠 Hard · asyncio with bounded concurrency

**Working solution**

In [ ]:
import asyncio

async def fetch_all(fetch, ids, concurrency=10):
    semaphore = asyncio.Semaphore(concurrency)

    async def bounded(item):
        async with semaphore:
            return await fetch(item)

    return await asyncio.gather(*(bounded(i) for i in ids), return_exceptions=True)

**Explanation**

asyncio runs thousands of waiting tasks on one thread; `await` is where a task yields control. A semaphore caps in-flight requests (rate limits, connection pools).
`gather` keeps input order regardless of completion order. Use `aiohttp`/`httpx` for real async HTTP — `requests` is blocking and would freeze the event loop.

**Complexity:** ≈ ceil(n / concurrency) × latency.

**Common mistakes**

- Calling blocking `requests.get` or `time.sleep` inside a coroutine.
- Unbounded `gather` over 100k URLs → server bans / socket exhaustion.

## Challenge — 🔴 Very Hard · Producer–consumer pipeline with bounded queues

**Working solution**

In [ ]:
import queue
import threading

_STOP = object()

def run_threaded_pipeline(items, stages, workers_per_stage=4, queue_size=100):
    queues = [queue.Queue(maxsize=queue_size) for _ in range(len(stages) + 1)]
    errors, errors_lock = [], threading.Lock()
    threads = []

    def worker(stage_index, func, inbox, outbox, remaining):
        while True:
            packet = inbox.get()
            if packet is _STOP:
                with remaining["lock"]:
                    remaining["n"] -= 1
                    last = remaining["n"] == 0
                if last:
                    downstream = workers_per_stage if stage_index + 1 < len(stages) else 1
                    for _ in range(downstream):
                        outbox.put(_STOP)
                return
            original, value = packet
            try:
                outbox.put((original, func(value)))
            except Exception as exc:
                with errors_lock:
                    errors.append((original, stage_index, f"{type(exc).__name__}: {exc}"))

    for index, func in enumerate(stages):
        remaining = {"n": workers_per_stage, "lock": threading.Lock()}
        for _ in range(workers_per_stage):
            t = threading.Thread(target=worker, args=(index, func, queues[index], queues[index + 1], remaining), daemon=True)
            t.start()
            threads.append(t)

    results = []
    def collector():
        while (packet := queues[-1].get()) is not _STOP:
            results.append(packet[1])
    sink = threading.Thread(target=collector, daemon=True)
    sink.start()

    for item in items:
        queues[0].put((item, item))
    for _ in range(workers_per_stage if stages else 1):
        queues[0].put(_STOP)
    for t in threads:
        t.join()
    sink.join()
    return results, errors

**Explanation**

Bounded queues give natural backpressure; sentinels give deterministic shutdown without timeouts or polling; a per-stage countdown ensures downstream stops only after *all* upstream
workers are done. This is the architecture of most ingestion services (and what Spark's executors, Kafka consumers, and Glue's streaming jobs manage for you).

**Complexity:** Throughput ≈ slowest stage × its workers.

**Common mistakes**

- Unbounded queues → memory blow-up when a later stage is slow.
- Sending only one sentinel for N workers → hangs forever.
- Daemon threads with no join → lost results.